# PoseNet Post-Training Quantization in PyTorch using the Model Compression Toolkit(MCT)

## Overview
This quick-start guide explains how to use the **Model Compression Toolkit (MCT)** to quantize a PoseNet model. We will load a pre-trained model and quantize it using the MCT with **Post-Training Quantization (PTQ)**. Finally, we will evaluate the quantized model and export it to an ONNX file.  

## Summary
In this tutorial, we will cover:

1. Loading and preprocessing COCO’s validation dataset.
2. Constructing an unlabeled representative dataset.
3. Post-Training Quantization using MCT.
4. Accuracy evaluation of the floating-point and the quantized models.

## posenet-pytorch(Independent library)
[posenet-pytorch](https://github.com/michellelychan/posenet-pytorch)  
This tutorial uses the unofficial repository linked below. Installation instructions are provided in the **Setup** section.  
This repository accesses Google's TensorFlow.js version of the PoseNet model and converts the retrieved model into a PyTorch model.  
The model uses MobileNetV1 as its backbone.  
You can choose from four model depths: 50, 75, 100, and 101.Model selection can be configured in the **Parameter setting** section described later.  

### Reference
Copyright 2018 Ross Wightman

Licensed under the Apache License, Version 2.0 (the "License");  
you may not use this file except in compliance with the License.  
You may obtain a copy of the License at  

http://www.apache.org/licenses/LICENSE-2.0

## Setup 
First, clone the GitHub repository.
This repository is the unofficial repository mentioned earlier.

In [ ]:
import os

if not os.path.isdir('posenet-pytorch'):
    !git clone https://github.com/michellelychan/posenet-pytorch.git

In the `__init__.py` file within the cloned repository (posenet-pytorch/posenet/\_\_init\_\_.py), the function `decode_multiple_poses` is currently disabled by being commented out. Therefore, enable it using the following command:

In [ ]:
!sed -i '2s/^# *\(from .* import .*\)/\1/' ./posenet-pytorch/posenet/__init__.py

```python
# ./posenet-pytorch/posenet/__init__.py
from posenet.constants import *
from posenet.decode_multi import decode_multiple_poses  # <-- this sentence
from posenet import decode
from posenet.models.model_factory import load_model
from posenet.models import MobileNetV1, MOBILENET_V1_CHECKPOINTS
from posenet.utils import *
```

Install the relevant packages:  
This step may take several minutes...


In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install onnx==1.16.1
!pip install numpy==1.26.4
!pip install opencv-python==4.9.0.80
!pip install pycocotools==2.0.10

In [ ]:
import importlib
if not importlib.util.find_spec('model_compression_toolkit'):
    !pip install model_compression_toolkit

In [ ]:
import itertools
import json
from typing import List, Dict, Any
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os
import sys
sys.path.append('./posenet-pytorch')
sys.path.append('./posenet-pytorch/posenet')
import posenet
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

### Various Settings
Here, you can configure the parameters listed below.  

#### File path setting
- SAVE_FLOAT_EVAL_RESULT  
  This parameter sets the filename for outputting inference results before quantization.
- SAVE_QUANT_EVAL_RESULT  
  This parameter sets the filename for outputting inference results before quantization.
- SAVE_QUANT_PATH  
  This parameter specifies the output destination for the quantized model.
- SAVE_QUANT_NAME  
  This parameter specifies the output filename for the quantization model.

#### Parameter setting
- MODEL_ID  
  This parameter allows you to select the model depth to use(50, 75, 100, 101).
- SCALE_FACTOR  
  This parameter allows you to set the scaling for the input image.
- DECODE_MAX_POSES  
  This parameter allows you to set the maximum number of detections in pose estimation.
- DECODE_MIN_POSE_SCORE  
  This parameter allows you to set the minimum score for pose detection.
- KPT_VIS_THR  
  This parameter allows you to set the visibility of keypoints.
- NUM_WORKERS  
  This parameter allows you to set the number of processes for parallelizing the data loading process.
- CALIB_ITER  
  This parameter allows you to set how many samples to use when generating representative data for quantization.

In [ ]:
# File path setting
SAVE_FLOAT_EVAL_RESULT = "float_eval_result"
SAVE_QUANT_EVAL_RESULT = "quant_eval_result"
SAVE_QUANT_PATH = "./_models/"
SAVE_QUANT_NAME = "qmodel"

# Parameter setting
MODEL_ID = 100
SCALE_FACTOR = 1.0
DECODE_MAX_POSES = 20
DECODE_MIN_POSE_SCORE = 0
KPT_LAB_THR = 0.2
KPT_VIS_THR = 0.5
NUM_WORKERS = 8
CALIB_ITER = 100

Load a pre-trained PoseNet(MobileNetV1 backbone) model.  

In [ ]:
float_model = posenet.load_model(MODEL_ID)
output_stride = getattr(float_model, 'output_stride', 8)

**Note**  
When you run the code for the first time, the model download will begin.  
This step may take several minutes...

## Dataset preparation
### Download COCO validation set
Download COCO dataset.

**Note**  
That for demonstration purposes we use the validation set for the model quantization routines. Usually, a subset of the training dataset is used, but loading it is a heavy procedure that is unnecessary for the sake of this demonstration.

This step may take several minutes...

In [ ]:
if not os.path.isdir('COCO_dataset'):
    !mkdir COCO_dataset
    !wget -P COCO_dataset http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !wget -P COCO_dataset http://images.cocodataset.org/zips/val2017.zip
    !unzip COCO_dataset/annotations_trainval2017.zip -d COCO_dataset
    !unzip COCO_dataset/val2017.zip -d COCO_dataset

Here, we are setting the paths for the annotation file and image folder of the downloaded dataset.

In [ ]:
COCO_IMG_DIR = "COCO_dataset/val2017/"
COCO_ANN_JSON = "COCO_dataset/annotations/person_keypoints_val2017.json"

Here, we read images and associated information from the configured COCO image folder and process them for use in the validation data.

In [ ]:
class CocoPoseNetValDataset(Dataset):
    def __init__(self, img_dir: str, ann_json: str, output_stride: int, scale_factor: float = 1.0):
        self.img_dir = img_dir
        self.coco = COCO(ann_json)
        self.img_ids = self.coco.getImgIds(catIds=[1])
        self.output_stride = output_stride
        self.scale_factor = scale_factor

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        input_image, draw_image, output_scale = posenet.read_imgfile(
            img_path, scale_factor=self.scale_factor, output_stride=self.output_stride
        )
        input_tensor = torch.from_numpy(input_image)

        sample = {
            'input': input_tensor,
            'img_id': img_id,
            'output_scale': output_scale,
            'file_name': img_info['file_name'],
        }
        return sample

Generate an array of keypoints with visibility/invisibility flags based on keypoint information and their scores.

In [ ]:
def coco_kpts_xy_score_to_xyv(
    kpts_xy: np.ndarray, kpts_score: np.ndarray,
    visible_thr: float = 0.5, label_thr: float = 0.2) -> np.ndarray:

    # COCO v Format Extension: Assign 0/1/2 based on score
    # visible_thr: Threshold for determining visibility
    # label_thr: Threshold for determining presence of label
    
    # v=0: does not exist within the image (< label_thr)
    vis = np.zeros_like(kpts_score, dtype=np.int32)
    # v=2: Completely visible (> visible_thr)
    vis[kpts_score > visible_thr] = 2
    # v=1: The label is present but not visible within the image (> label_thr, < visible_thr)
    vis[(kpts_score > label_thr) & (kpts_score <= visible_thr)] = 1
    # Combination to [x, y, v]
    kpts_xyv = np.concatenate([kpts_xy, vis[:, None]], axis=1)
    return kpts_xyv

Organize key point information into a one-dimensional list.

In [ ]:
def flatten_xyv(kpts_xyv: np.ndarray) -> List[float]:
    return [float(v) for row in kpts_xyv for v in row]

In [ ]:
val_dataset = CocoPoseNetValDataset(
    img_dir=COCO_IMG_DIR, ann_json=COCO_ANN_JSON,
    output_stride=output_stride, scale_factor=SCALE_FACTOR
)

# For evaluation (batch size 1)
val_dataloader = DataLoader(
    val_dataset, batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=lambda x: x[0]
)

# For calibration（No label required）
calib_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=True,
    num_workers=max(1, NUM_WORKERS // 2), collate_fn=lambda x: x[0]
)
print(len(val_dataset))

## Representative Dataset
For quantization with MCT, we need to define a representative dataset required by the PTQ algorithm. This dataset is a generator that returns a list of images:

In [ ]:
def representative_dataset_gen():
    for sample in itertools.islice(itertools.cycle(calib_loader), CALIB_ITER):
        yield [sample['input']]

## Target Platform Capabilities (TPC)
In addition, MCT optimizes the model for dedicated hardware platforms. This is done using TPC (for more details, please visit our [documentation](https://sonysemiconductorsolutions.github.io/mct-model-optimization/api/api_docs/modules/target_platform_capabilities.html)). Here, we use the default Pytorch TPC:

In [ ]:
import model_compression_toolkit as mct

tpc = mct.get_target_platform_capabilities('pytorch', 'default')

# Post-Training Quantization using MCT
Now for the exciting part! Let's run PTQ on the model.

In [ ]:
quantized_model, quantization_info = mct.ptq.pytorch_post_training_quantization(
                                        in_module=float_model,
                                        representative_data_gen=representative_dataset_gen,
                                        target_platform_capabilities=tpc)

# Model Evaluation
Now, we will create a function for evaluating a model.  
The inference results before and after quantization are displayed on the terminal and simultaneously written to a JSON file.

In [ ]:
@torch.no_grad()
def evaluate(model: torch.nn.Module,
             val_dataset: CocoPoseNetValDataset,
             val_dataloader: DataLoader,
             decode_max_poses: int = 1,
             decode_min_pose_score: float = 0,
             kpt_lab_thr: float = 0.2,
             kpt_vis_thr: float = 0.5) -> float:

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    output_stride = val_dataset.output_stride

    results = []
    for sample in tqdm(val_dataloader, desc="Evaluating"):
        inp = sample['input'].to(device)
        img_id = sample['img_id']
        output_scale = sample['output_scale']

        heat, off, disp_f, disp_b = model(inp)
        heat, off, disp_f, disp_b = heat.squeeze(0), off.squeeze(0), disp_f.squeeze(0), disp_b.squeeze(0)

        # decode
        pose_scores, keypoint_scores, keypoint_coords, pose_offsets = posenet.decode_multiple_poses(
            heat,
            off,
            disp_f,
            disp_b,
            output_stride=output_stride,
            max_pose_detections=decode_max_poses,
            min_pose_score=decode_min_pose_score)

        for p_idx, ps in enumerate(pose_scores):
            if ps == 0.0:
                continue
            kpts_xy = keypoint_coords[p_idx]
            kpts_xy_img = np.zeros_like(kpts_xy)
            kpts_xy_img[:, 0] = kpts_xy[:, 1]*output_scale[1]
            kpts_xy_img[:, 1] = kpts_xy[:, 0]*output_scale[0]
            kpts_sc = keypoint_scores[p_idx]
            kpts_xyv = coco_kpts_xy_score_to_xyv(kpts_xy_img, kpts_sc, visible_thr=kpt_vis_thr, label_thr=kpt_lab_thr)
            keypoint = flatten_xyv(kpts_xyv)
            results.append({
                "image_id": int(img_id),
                "category_id": 1,
                "keypoints": keypoint,
                "score": float(ps)
            })

    if len(results) == 0:
        print("WARNING : No detection results found. Returning AP=0.0.")
        return
    
    if model==float_model:
        with open(os.path.join(SAVE_FLOAT_EVAL_RESULT + '.json'), 'w') as f:
            json.dump(results,f,ensure_ascii=False,indent=1)
    else:
        with open(os.path.join(SAVE_QUANT_EVAL_RESULT + '.json'), 'w') as f:
            json.dump(results,f,ensure_ascii=False,indent=1)

    # 評価
    coco_gt = val_dataset.coco
    coco_dt = coco_gt.loadRes(results)
    evaluator = COCOeval(coco_gt, coco_dt, iouType='keypoints')
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    ap = float(evaluator.stats[0])
    print(f"AP (OKS mAP): {ap:.4f}")

Lets's start with the floating-point model evaluation.  
This step may take several minutes...

In [ ]:
print("evaluating float model（COCO mAP）...")
evaluate(float_model,
            val_dataset,
            val_dataloader,
            decode_max_poses=DECODE_MAX_POSES,
            decode_min_pose_score=DECODE_MIN_POSE_SCORE,
            kpt_vis_thr=KPT_VIS_THR,
            kpt_lab_thr=KPT_LAB_THR)

Finally, let's evaluate the quantized model:  
This step may take several minutes...

In [ ]:
print("evaluating quantized model（COCO mAP）...")
evaluate(quantized_model,
            val_dataset,
            val_dataloader,
            decode_max_poses=DECODE_MAX_POSES,
            decode_min_pose_score=DECODE_MIN_POSE_SCORE,
            kpt_vis_thr=KPT_VIS_THR,
            kpt_lab_thr=KPT_LAB_THR)

# Model Export
Export the quantized model in ONNX format.

In [ ]:
mct.exporter.pytorch_export_model(quantized_model, save_model_path=os.path.join(SAVE_QUANT_PATH + SAVE_QUANT_NAME + '.onnx'), repr_dataset=representative_dataset_gen)

## Conclusion
This tutorial demonstrated how to quantize the PoseNet skeleton detection model using MCT in a hardware-friendly manner. 
  
The key advantage of hardware-friendly quantization is that the model can run more efficiently in terms of runtime, power consumption, and memory usage on designated hardware.

MCT can deliver competitive results across a wide range of tasks and network architectures. For more details, [check out the paper:](https://arxiv.org/abs/2109.09113).

## Copyrights

Copyright 2025 Sony Semiconductor Solutions, Inc. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
